# 02 · Preprocesamiento: del archivo consolidado a las particiones

Este cuaderno documenta y ejecuta la capa de datos del proyecto (`src/data_processing.py` sobre `genial_clf.data`, `genial_clf.split` y `genial_clf.coder`). La exploración descriptiva está en `01_exploracion.ipynb`; aquí interesa **qué transformaciones sufre el corpus antes de llegar al modelo y por qué**.

1. Carga con verificación de integridad (SHA-256).
2. Normalización de categorías y correcciones acordadas con el equipo experto.
3. Construcción del texto de entrada (`título. observaciones`) y limpieza.
4. Detección de idioma y marcado de duplicados por URL.
5. Exclusión de columnas que filtrarían la etiqueta (`Investigador`, `Tipo de norma`, `Fecha`).
6. Partición entrenamiento / pool *few-shot* / prueba ciega, con las etiquetas de la prueba ocultas.

In [1]:
import json, os, subprocess, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
from utils import load_config, setup_logging, set_seed, CLASSES
cfg = load_config(); set_seed(cfg["seed"]); setup_logging()
RES = ROOT / "results/metrics"; FIG = ROOT / "results/figures"; RES.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
print("raíz del repositorio:", ROOT)

raíz del repositorio: /home/claude/repo_genial


## 1. Carga y verificación de integridad

El corpus autorizado (813 normas × 13 columnas) no se versiona. El hash fijado en `config.yaml` evita trabajar sin saberlo sobre una versión distinta del Excel.

In [2]:
from utils import sha256_of
from genial_clf import data as D
raw = ROOT / cfg["paths"]["raw_corpus"]
h = sha256_of(raw); print("SHA-256:", h, "\ncoincide con config.yaml:", h == cfg["corpus_sha256"])
bruto = D.load_raw(raw, cfg["paths"]["raw_sheet"], cfg["corpus_sha256"])
print("filas × columnas:", bruto.shape); print("columnas:", list(bruto.columns))

SHA-256: f9797147e7026f796d453fa84d09c4bf32c48ae75c3744912806a72db3ff7f33 
coincide con config.yaml: True


2026-10-11 04:52:06,493 · INFO · genial_clf.data · corpus cargado: 813 filas × 13 columnas


filas × columnas: (813, 13)
columnas: ['Investigador', 'País', 'Título de la Norma', 'Tipo de norma', 'Estado', 'Organismo Emisor/Universidad', 'Dominio', 'Vínculo con Educación', 'Dedicación del Texto', 'Fecha de Publicación', 'URL Oficial', 'Observaciones', 'Ámbito']


## 2-4. Preprocesamiento

`preprocess()` aplica en orden: normalización de `Tipo de norma`, correcciones expertas (etiquetas revisadas con el equipo GENIAL), construcción de `texto`, conteo de palabras, detección de idioma (biblioteca *lingua*, con el país como apoyo cuando el título es ambiguo) y `grupo_url` (normas que comparten URL oficial se tratan como una sola unidad al particionar).

In [3]:
df = D.preprocess(bruto.copy(), cfg)
t = cfg["columns"]["target"]
print("columnas añadidas:", [c for c in df.columns if c not in bruto.columns])
print("\nDistribución de la variable objetivo:"); display(df[t].value_counts().to_frame("n").assign(pct=lambda x: (100 * x["n"] / len(df)).round(1)))
print("idioma del título:", df["idioma"].value_counts().to_dict())
print("grupos de URL con más de una norma:", int((df.groupby("grupo_url").size() > 1).sum()), "· normas implicadas:", int(df.groupby("grupo_url")["id_norma"].transform("size").gt(1).sum()))
print("longitud del texto (palabras): mediana %.0f · media %.1f · p10 %.0f · p90 %.0f" % (df["n_palabras"].median(), df["n_palabras"].mean(), df["n_palabras"].quantile(.1), df["n_palabras"].quantile(.9)))

2026-10-11 04:52:06,517 · INFO · genial_clf.data · corrección experta aplicada: Ecuador / Políticas de Privacidad -> Protección de Datos (1 fila)


2026-10-11 04:52:07,644 · INFO · genial_clf.data · grupos de URL repetida: 14


columnas añadidas: ['texto', 'n_palabras', 'idioma', 'grupo_url', 'id_norma']

Distribución de la variable objetivo:


,n,pct
Dominio,,
Pedagógico,294,36.2
Ética,265,32.6
Administrativo,120,14.8
Protección de Datos,71,8.7
Técnico,63,7.7


idioma del título: {'es': 712, 'pt': 76, 'en': 25}
grupos de URL con más de una norma: 14 · normas implicadas: 30
longitud del texto (palabras): mediana 44 · media 50.9 · p10 25 · p90 88


### Ejemplo de texto de entrada

El modelo solo ve `texto`. Se muestran títulos (documentos públicos) con las observaciones truncadas.

In [4]:
ej = df.sample(3, random_state=cfg["seed"])[["id_norma", "País", t, "idioma", "n_palabras", "texto"]].copy()
ej["texto"] = ej["texto"].str.slice(0, 160) + "…"; display(ej)

,id_norma,País,Dominio,idioma,n_palabras,texto
230,231,México,Pedagógico,es,44,Programa de estudios de Técnico en Inteligenci...
134,135,México,Ética,es,35,Lineamientos para la implementación de la Inte...
246,247,Ecuador,Administrativo,es,62,Resolución Sesión Ordinaria (Normativa IA). La...


## 5. Variables excluidas por fuga de información

`Investigador` (quién codificó) tiene V de Cramér 0.99 con `País`: cada investigador cubre países concretos, de modo que la columna equivale a conocer el país y, además, es un dato personal. `Tipo de norma` y `Fecha de Publicación` no describen el contenido. Ninguna entra al modelo; `Investigador` además se anonimiza (Codificador A-J) en cualquier artefacto del repositorio.

In [5]:
print("excluidas del modelo:", cfg["columns"]["leakage_excluded"])
probe = D.leakage_probe(df, cfg)
print(f"fuga léxica (la etiqueta aparece literalmente en el texto): {len(probe)} normas de {len(df)} ({100*len(probe)/len(df):.1f} %) → se conservan; es vocabulario legítimo de la norma, no una columna filtrada")
display(probe.assign(texto=probe["texto"].str.slice(0, 110) + "…").head(5))

excluidas del modelo: ['Tipo de norma', 'Investigador', 'Fecha de Publicación']
fuga léxica (la etiqueta aparece literalmente en el texto): 157 normas de 813 (19.3 %) → se conservan; es vocabulario legítimo de la norma, no una columna filtrada


,id_norma,Dominio,texto
9,10,Protección de Datos,"Data Protection Act, 2020 (Ley de Protección d..."
23,24,Protección de Datos,Acuerdo No. 12 - Política de Protección de Dat...
28,29,Ética,Resolución de Rectoría No. R-2025-022 - Políti...
32,33,Protección de Datos,Política de protección de datos. Herramienta f...
38,39,Ética,CONPES 4144 de 2025: Política Nacional de Inte...


## 6. Particiones

Si existe el instrumento de codificación ciega del investigador (`data/raw/codificacion_investigador_A.xlsx`), la **prueba ciega son exactamente sus normas** (119 emparejadas por título/URL + sus duplicados → 125) y las 23 normas nuevas del Caribe que no están en la versión congelada se guardan aparte. El resto se divide en pool *few-shot* (3 por clase, entre 25 y 90 palabras) y entrenamiento. Entrenamiento y prueba no comparten ningún `grupo_url`.

In [6]:
from data_processing import make_partitions, load_splits
splits, origen = make_partitions(df, cfg); print(origen)
for k in ("train", "fewshot", "test"):
    print(f"{k:8s} {len(splits[k]):4d}", splits[k][t].value_counts().to_dict() if k != "test" else "(etiquetas ocultas en test_labels_hidden.csv)")
assert not set(splits["train"]["grupo_url"]) & set(splits["test"]["grupo_url"]), "fuga por URL entre entrenamiento y prueba"
sp = load_splits(cfg); print("\narchivos escritos en", cfg["paths"]["splits_dir"], "→", {k: list(v.columns)[:6] for k, v in sp.items()})
print("idioma en prueba:", sp["test"]["idioma"].value_counts().to_dict())

2026-10-11 04:52:08,319 · INFO · genial_clf.split · archivos escritos en /home/claude/repo_genial/data/splits


2026-10-11 04:52:08,320 · INFO · data_processing · particiones: instrumento de codificación ciega del investigador A (119 normas en la versión congelada; 23 nuevas fuera de ella) · train 673 · few-shot 15 · test 125


instrumento de codificación ciega del investigador A (119 normas en la versión congelada; 23 nuevas fuera de ella)
train     673 {'Pedagógico': 246, 'Ética': 219, 'Administrativo': 100, 'Protección de Datos': 58, 'Técnico': 50}
fewshot    15 {'Pedagógico': 3, 'Ética': 3, 'Técnico': 3, 'Protección de Datos': 3, 'Administrativo': 3}
test      125 (etiquetas ocultas en test_labels_hidden.csv)

archivos escritos en /home/claude/repo_genial/data/splits → {'train': ['Investigador', 'País', 'Título de la Norma', 'Tipo de norma', 'Estado', 'Organismo Emisor/Universidad'], 'fewshot': ['id_norma', 'País', 'idioma', 'texto', 'Dominio'], 'test': ['id_norma', 'País', 'idioma', 'texto']}
idioma en prueba: {'es': 112, 'pt': 12, 'en': 1}


## Comprobaciones de calidad

`results/metrics/reporte_calidad.md` recoge faltantes, duplicados y distribución final; es el informe que acompaña a `data/README.md`.

In [7]:
print((RES / "reporte_calidad.md").read_text(encoding="utf-8")[:2500])

# Reporte de calidad del corpus

- Filas: 813 · Columnas originales: 13 · Texto medio: 50.9 palabras (mediana 44)
- Distribución de la variable objetivo:
  - Pedagógico: 294 (36.2 %)
  - Ética: 265 (32.6 %)
  - Administrativo: 120 (14.8 %)
  - Protección de Datos: 71 (8.7 %)
  - Técnico: 63 (7.7 %)
- Razón de desbalance: 4.7:1
- Idioma (detección automática con respaldo por país):
  - es: 712
  - pt: 76
  - en: 25
- Países: 28 · los 5 primeros concentran 67 %
- Grupos de URL repetida: 14
- Categorías de `Tipo de norma` tras normalizar: 11
- Variables excluidas como predictoras por diseño: Tipo de norma, Investigador, Fecha de Publicación

